In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df_renewal = pd.read_csv('../../../data/interim/renewal_calls_cleaned.csv')
df_email = pd.read_csv('../../../data/interim/emails_cleaned.csv')
df_cc = pd.read_csv('../../../data/interim/cc_calls_cleaned.csv')
df_billings = pd.read_csv('../../../data/interim/billings_cleaned.csv')

In [4]:
df_cc.info()

<class 'pandas.DataFrame'>
RangeIndex: 31636 entries, 0 to 31635
Data columns (total 30 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   contact_id                                31636 non-null  float64
 1   call_date                                 31636 non-null  str    
 2   direction                                 31636 non-null  str    
 3   cc_care_package                           31636 non-null  str    
 4   cc_care_package_discussed                 31636 non-null  str    
 5   cc_urgency_getting_on_site                31636 non-null  str    
 6   cc_external_consultant                    31636 non-null  str    
 7   cc_agent_cross_sell_attempt               31636 non-null  str    
 8   cc_customer_issues_concerns               31636 non-null  str    
 9   cc_business_struggles_financial_hardship  31636 non-null  str    
 10  cc_call_initiated_by                      316

In [5]:
df_merged = (
    df_billings
    .merge(df_renewal, on='co_ref', how='left', suffixes=('', '_renewal'))
    .merge(df_email, on='co_ref', how='left', suffixes=('', '_email'))
    .merge(df_cc, on='co_ref', how='left', suffixes=('', '_cc'))
)

In [6]:
df_merged.shape

(668008, 130)

In [7]:
for col in df_merged.columns:
    print(df_merged[col].value_counts(dropna=False))
    print(df_merged[col].isnull().sum())
    print("-" * 50)

co_ref
VU9662    3968
AD1657    1610
DB3207    1292
AA0994    1287
FW0890     900
          ... 
XQ5674       1
MH5611       1
LQ9360       1
EX0639       1
XU6585       1
Name: count, Length: 47814, dtype: int64
0
--------------------------------------------------
renewal_month
2025-10-01    21429
2025-11-01    21214
2024-11-01    20904
2024-10-01    20506
2025-09-01    19763
2024-09-01    19288
2024-06-01    19064
2025-01-01    18402
2024-05-01    18228
2025-06-01    18111
2025-05-01    18062
2023-11-01    18056
2023-10-01    17873
2026-01-01    17650
2025-07-01    17613
2026-02-01    17569
2024-07-01    17436
2023-09-01    16961
2025-02-01    16928
2024-08-01    16836
2023-06-01    16728
2025-08-01    16650
2026-05-01    16192
2023-05-01    16080
2024-01-01    15590
2023-07-01    15499
2024-02-01    14685
2024-04-01    14473
2023-08-01    14292
2025-04-01    14225
2025-12-01    13923
2023-01-01    13903
2024-12-01    13762
2025-03-01    13443
2023-02-01    13351
2023-04-01    12994


registration_date
2019-03-25    5813
2019-05-08    4112
2025-02-18    1649
2019-04-16    1635
2017-01-25    1411
              ... 
2024-10-05       1
2024-10-06       1
2024-07-27       1
2024-11-02       1
2024-12-15       1
Name: count, Length: 5787, dtype: int64
0
--------------------------------------------------
proforma_account_stage
Published          412736
Membership Only    103043
Renewal Process     93415
Unknown             38489
Vetting             14318
Suspended            5995
Retired                12
Name: count, dtype: int64
0
--------------------------------------------------
proforma_audit_status
Accredited                                                  406644
Failed- Renewal Questionnaire not received                   43606
Renewal Questionnaire Received                               40865
Unknown                                                      38489
Renewal Questionnaire Sent                                   24119
Failed- Renewal Additional Information 

proforma_membership_status
Accredited     476732
Member Only    116314
In Progress     74949
Non Member         13
Name: count, dtype: int64
0
--------------------------------------------------
proforma_approved_lists
1      201363
0      136023
2      103313
3       58093
4       39101
        ...  
171         1
130         1
162         1
156         1
137         1
Name: count, Length: 135, dtype: int64
0
--------------------------------------------------
tenure_years
1     84334
2     74855
3     60512
4     52374
5     51055
6     41858
7     39729
8     35607
9     32123
10    28834
11    24042
12    21343
13    19449
14    18324
15    18216
16    16225
17    11723
18     8310
20     8250
19     7597
21     6317
22     4315
0      1589
23      799
24      153
25       71
26        4
Name: count, dtype: int64
0
--------------------------------------------------
band
Band B     174100
Band C1    130048
Band D     102755
Band C2     86650
Band E      53272
Band F1     33888
Band F2

0
--------------------------------------------------
last_renewal
2024-11-01    19030
2024-10-01    18641
2023-11-01    18223
2023-10-01    17904
2024-09-01    17518
              ...  
2025-06-22        1
2025-04-28        1
2025-12-27        1
2025-07-27        1
2025-11-02        1
Name: count, Length: 1155, dtype: int64
0
--------------------------------------------------
last_band
Band B     176683
Band C1    130011
Band D     100940
Band C2     89839
Band E      51013
Band F1     25428
Band F      24191
Band A      19833
Band G      16755
Band F2     15707
Group        6045
Band H       5151
Band I       3645
Band J       2767
Name: count, dtype: int64
0
--------------------------------------------------
last_total_net_paid
539.0     28209
579.0     25316
999.0     22613
684.0     22592
949.0     20335
          ...  
184.0         1
872.8         1
1109.2        1
3096.2        1
4589.0        1
Name: count, Length: 1289, dtype: int64
0
------------------------------------------

membership_renewal_decision
No         348456
Unknown    264729
NaN         29278
Yes         25545
Name: count, dtype: int64
29278
--------------------------------------------------
serious_complaint
No         361796
Unknown    274723
NaN         29278
Yes          2211
Name: count, dtype: int64
29278
--------------------------------------------------
other_complaint
No         278697
Unknown    274737
Yes         85296
NaN         29278
Name: count, dtype: int64
29278
--------------------------------------------------
discussion_on_price_increase
No         344093
Unknown    258020
Yes         36617
NaN         29278
Name: count, dtype: int64
29278
--------------------------------------------------
renewal_impact_due_to_price_increase
No         366755
Unknown    258265
NaN         29278
Yes         13710
Name: count, dtype: int64
29278
--------------------------------------------------
discount_or_waiver_requested
No         368528
Unknown    258282
NaN         29278
Yes         11

...

238007
--------------------------------------------------
crm_accreditation_completed
NaN              238007
Not Discussed    228307
No               108963
Unknown           47419
Yes               45312
Name: count, dtype: int64
238007
--------------------------------------------------
crm_timely_completion
Not Discussed    345638
NaN              238007
Unknown           47419
No                28334
Yes                8610
Name: count, dtype: int64
238007
--------------------------------------------------
crm_progress_towards_accreditation
Not Discussed    290974
NaN              238007
Yes               81971
Unknown           47419
No                 9637
Name: count, dtype: int64
238007
--------------------------------------------------
crm_delays_in_accreditation
No               287375
NaN              238007
Yes               90025
Unknown           47419
Not Discussed      5182
Name: count, dtype: int64
238007
--------------------------------------------------
crm_contracto

238007
--------------------------------------------------
crm_membership_level
NaN               238007
Accredited        158537
Not Discussed     150360
In progress        86802
Unknown            30105
Not Accredited      2066
Members only        1622
Standard             462
Premier               47
Name: count, dtype: int64
238007
--------------------------------------------------
crm_platform_issues_raised
No               307442
NaN              238007
Not Discussed     73446
Unknown           30105
Yes               19008
Name: count, dtype: int64
238007
--------------------------------------------------
crm_agent_chased_contractor
Yes              268890
NaN              238007
No               129393
Unknown           30105
Not Discussed      1613
Name: count, dtype: int64
238007
--------------------------------------------------
crm_agent_chase_count
NaN     238007
1.0     176202
0.0     131032
2.0      95741
3.0      22004
4.0       3503
5.0       1092
6.0        362
7.0    

238007
--------------------------------------------------
crm_financial_hardship_mentioned
Not Discussed    303264
NaN              238007
No                67911
Unknown           32106
Yes               26720
Name: count, dtype: int64
238007
--------------------------------------------------
year
2025.0    302539
NaN       238007
2026.0    127462
Name: count, dtype: int64
238007
--------------------------------------------------
sentiment_score_missing
0.0    246046
NaN    238007
1.0    183955
Name: count, dtype: int64
238007
--------------------------------------------------
contact_id
NaN             300559
6.854460e+11      2761
6.914790e+11      2653
6.862780e+11      2640
6.916730e+11      2621
                 ...  
6.217010e+11         4
5.645430e+11         3
5.647580e+11         3
5.640720e+11         2
5.647570e+11         1
Name: count, Length: 495, dtype: int64
300559
--------------------------------------------------
call_date_cc
NaN           300559
2025-11-26      2894

300559
--------------------------------------------------
cc_questionnaire_completion
NaN        300559
No         294543
Yes         72442
Unknown       464
Name: count, dtype: int64
300559
--------------------------------------------------
cc_chasing_response
NaN        300559
No         271194
Yes         93403
Unknown      2852
Name: count, dtype: int64
300559
--------------------------------------------------
cc_issues_within_questionnaire
No         307343
NaN        300559
Yes         45870
Unknown     14236
Name: count, dtype: int64
300559
--------------------------------------------------
cc_login_issues
No         346227
NaN        300559
Yes         20580
Unknown       642
Name: count, dtype: int64
300559
--------------------------------------------------
cc_platform_issues
No         340017
NaN        300559
Yes         26904
Unknown       528
Name: count, dtype: int64
300559
--------------------------------------------------
cc_dissatisfaction_time_to_complete
No         3

300559
--------------------------------------------------
cc_contractor_sentiment_end_score
NaN      300559
90.0     146573
80.0     130716
50.0      61015
100.0      8776
60.0       7514
70.0       5400
0.0        2975
40.0       1727
95.0       1260
20.0        899
30.0        503
10.0         87
75.0          4
Name: count, dtype: int64
300559
--------------------------------------------------
cc_contractor_sentiment_overall_score
NaN      300559
80.0     174182
90.0     112350
85.0      26652
70.0      15842
50.0      11658
60.0       8203
100.0      6489
40.0       3073
65.0       1689
55.0       1610
95.0       1459
0.0        1057
20.0        922
30.0        853
75.0        612
92.0        481
45.0        164
10.0         93
25.0         60
Name: count, dtype: int64
300559
--------------------------------------------------
cc_pricing_mentioned
No         317709
NaN        300559
Yes         48156
Unknown      1584
Name: count, dtype: int64
300559
--------------------------------

In [8]:
df_merged

,co_ref,renewal_month,sustainability_score,total_renewal_score_new,last_years_price,auto_renewal_score,status_scores,anchoring_score,tenure_scores,proforma_auto_renewal,proforma_world_pay_token,proforma_date,current_anchorings,payment_timeframe,registration_date,proforma_account_stage,proforma_audit_status,current_auto_renewal_flag,current_world_pay_token,renewal_score_at_release,proforma_membership_status,proforma_approved_lists,tenure_years,band,prospect_renewal_date,starting_net,starting_vat,starting_gross,starting_membership_net,starting_package_net,starting_pqq_net,membership_net,package_net,pqqnet,prospect_outcome,payment_method,total_amount,tenure_group,last_renewal,last_band,last_total_net_paid,last_connections,anchor_group,renewal_year,is_first_year,has_discount,discount_pct,call_id,call_direction,call_date,membership_renewal_decision,serious_complaint,other_complaint,discussion_on_price_increase,renewal_impact_due_to_price_increase,discount_or_waiver_requested,call_reschedule_request,agent_flagged_membership_status_alert,agent_renewal_initiation,explicit_competitor_mention,explicit_switching_intent,mentioned_competitors,price_switching_mentioned,competitor_value_comparison,competitor_benefits_mentioned,topic_introduced_by,percentage_price_increase_mentioned,monetary_price_increase_mentioned,price_range_mentioned,customer_asked_for_justification,customer_response,desire_to_cancel,discount_offered,call_number,time_to_renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,crm_dts_or_ssip_mentioned,crm_customer_payment_intention,crm_competitors_mentioned,crm_membership_level,crm_platform_issues_raised,crm_agent_chased_contractor,crm_agent_chase_count,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,year,sentiment_score_missing,contact_id,call_date_cc,direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_call_initiated_by,cc_questionnaire_completion,cc_chasing_response,cc_issues_within_questionnaire,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_contractor_sentiment,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained
0,VT6174,2024-11-01,8.0,42.5,799.0,9,9,7.5,9.0,True,True,2024-09-06,0,0,2021-11-05,Published,Accredited,y,y,26.0,Accredited,1,3,Band C1,2024-11-05,919,183.8,1102.8,664,120,135,664.0,0.0,135,Won,Card,799.0,3,2023-11-01,Band B,664.0,1,1,2024,0,0,0.0,6.920000e+11,OUT_BOUND,2026-01-13,No,No,No,No,No,No,Yes,No,Yes,No,No,No,No,Not Applicable,Not Applicable,Not Discussed,No,No,Not Discussed,No,Neutral,Not Discussed,No,5.0,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,neutral,50.0,No,Yes,No,Accredited,No,No,0.0,Not Discussed,Not Discussed,0.0,Not Discussed,No,No,No,No,Not Discussed,2025.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,VT6174,2024-11-01,8.0,42.5,799.0,9,9,7.5,9.0,True,True,2024-09-06,0,0,2021-11-05,Published,Accredited,y,y,26.0,Accredited,1,3,Band C1,2024-11-05,919,183.8,1102.8,664,120,135,664.0,0.0,135,Won,Card,799.0,3,2023-11-01,Band B,664.0,1,1,2024,0,0,0.0,6.870000e+11,OUT_BOUND,2025-11-05,No,No,Yes,No,No,No,No,Yes,Yes,No,No,No,No,Not Applicable,Not Discussed,Not Discussed,No,No,Not Discussed,N

In [9]:

for col in df_merged.columns:
    if 'date' in col or 'month' in col:
        df_merged[col] = pd.to_datetime(df_merged[col])
        print(col)
        print(df_merged[col])
        

renewal_month
0        2024-11-01
1        2024-11-01
2        2024-11-01
3        2024-11-01
4        2024-11-01
            ...    
668003   2023-09-01
668004   2023-08-01
668005   2023-08-01
668006   2023-07-01
668007   2023-07-01
Name: renewal_month, Length: 668008, dtype: datetime64[us]
proforma_date
0        2024-09-06
1        2024-09-06
2        2024-09-06
3        2024-09-06
4        2024-09-06
            ...    
668003   2023-07-06
668004   2023-06-07
668005   2023-06-07
668006   2023-05-10
668007   2023-05-10
Name: proforma_date, Length: 668008, dtype: datetime64[us]
registration_date
0        2021-11-05
1        2021-11-05
2        2021-11-05
3        2021-11-05
4        2021-11-05
            ...    
668003   2020-09-15
668004   2020-08-03
668005   2020-08-03
668006   2020-07-08
668007   2020-07-08
Name: registration_date, Length: 668008, dtype: datetime64[us]
prospect_renewal_date
0        2024-11-05
1        2024-11-05
2        2024-11-05
3        2024-11-05
4        20

In [10]:
open_prospects = df_merged[df_merged['prospect_outcome'] == 'Open']

In [11]:
open_prospects['prospect_outcome'].value_counts()

prospect_outcome
Open    37604
Name: count, dtype: int64

In [12]:
df_merged.shape

(668008, 130)

In [13]:
df_filtered = df_merged[
    (
        # Group all date logic together first
        (
            ((df_merged['prospect_renewal_date'] - df_merged['call_date']).dt.days.between(0, 14)) |
            ((df_merged['prospect_renewal_date'] - df_merged['call_date_cc']).dt.days.between(0, 14))
        )
    ) 
    & 
    (df_merged['prospect_outcome'] != 'Open') # Now applies to both date conditions
]

In [14]:
df_filtered.shape

(78325, 130)

In [15]:
print(df_filtered['prospect_outcome'].value_counts())

prospect_outcome
Won        68967
Churned     9358
Name: count, dtype: int64


In [16]:
for col in df_filtered.columns:
    print(df_filtered[col].value_counts(dropna=False))
    print(df_filtered[col].isnull().sum())
    print("-" * 50)

co_ref
AA0994    243
RE6328    102
VU9662     93
GK6414     80
XB6908     78
         ... 
MR9914      1
AH3239      1
NS9452      1
IK6353      1
HG7200      1
Name: count, Length: 21206, dtype: int64
0
--------------------------------------------------
renewal_month
2025-02-01    4530
2025-10-01    4279
2025-09-01    4257
2025-01-01    4216
2024-10-01    4128
2024-11-01    4121
2025-11-01    4069
2024-09-01    3880
2024-07-01    3730
2024-05-01    3675
2025-03-01    3614
2024-08-01    3423
2025-07-01    3275
2025-08-01    3208
2024-06-01    3161
2024-12-01    2920
2025-06-01    2902
2024-04-01    2711
2025-05-01    2673
2025-12-01    2548
2026-01-01    2516
2025-04-01    2473
2026-02-01    2016
Name: count, dtype: int64
0
--------------------------------------------------
sustainability_score
8.0    44439
9.5    24068
9.0     9818
Name: count, dtype: int64
0
--------------------------------------------------
total_renewal_score_new
44.0    9954
43.0    7409
45.5    7115
45.0    6731


17879


--------------------------------------------------
crm_contractor_sentiment_score
 50.0     27170
-1.0      21710
 NaN      17879
 80.0      5039
 20.0      2235
 90.0      1439
 30.0       894
 40.0       742
 0.0        651
 100.0      507
 16.0        18
 98.0        12
 84.0         9
 48.0         5
 55.0         5
 60.0         3
 31.0         3
 25.5         2
 70.0         1
 91.0         1
Name: count, dtype: int64
17879
--------------------------------------------------
crm_dts_or_ssip_mentioned
No               41820
NaN              17879
Yes              13064
Unknown           5561
Not Discussed        1
Name: count, dtype: int64
17879
--------------------------------------------------
crm_customer_payment_intention
Not Discussed    25887
Yes              23151
NaN              17879
No                5847
Unknown           5561
Name: count, dtype: int64
17879
--------------------------------------------------
crm_competitors_mentioned
No               45735
NaN         

In [17]:
# Columns to remove: identifiers, dates used for filtering, derived scores that may cause leakage, proforma statuses, and other non-beneficial columns
columns_to_drop = [
    'co_ref', 'renewal_month', 'contact_id', 'call_id', 'proforma_date', 'call_date', 'call_date_cc', 
    'registration_date', 'prospect_renewal_date', 'starting_vat', 'starting_gross',
    'proforma_world_pay_token', 'current_world_pay_token', 'proforma_audit_status',
    'proforma_approved_lists', 'proforma_account_stage', 'sentiment_score_missing',
    'year', 'renewal_year', 'direction'
]

df_filtered = df_filtered.drop(columns=columns_to_drop)

In [18]:
num_cols = []

for col in df_filtered.columns:
    if df_filtered[col].dtype in ['int64', 'float64']:
        print(f"Column: {col}")
        num_cols.append(col)
        print(df_filtered[col].value_counts())
        print(f"Null values: {df_filtered[col].isnull().sum()}")
        print("-" * 50)

Column: sustainability_score
sustainability_score
8.0    44439
9.5    24068
9.0     9818
Name: count, dtype: int64
Null values: 0
--------------------------------------------------
Column: total_renewal_score_new
total_renewal_score_new
44.0    9954
43.0    7409
45.5    7115
45.0    6731
44.5    6652
43.5    6314
42.0    5026
42.5    3345
41.0    3198
41.5    2544
46.5    2364
32.0    1972
33.0    1883
46.0    1783
34.0    1454
32.5    1439
40.5    1438
40.0    1397
35.0    1125
33.5    1007
34.5     937
31.0     937
39.5     754
31.5     704
39.0     315
38.5     166
36.0     133
35.5      77
30.5      59
36.5      55
38.0      29
37.5       8
37.0       1
Name: count, dtype: int64


Null values: 0
--------------------------------------------------
Column: last_years_price
last_years_price
599.00     3454
999.00     3427
799.00     2901
949.00     2341
579.00     2242
           ... 
702.00        1
813.60        1
833.40        1
1592.10       1
258.75        1
Name: count, Length: 1416, dtype: int64
Null values: 0
--------------------------------------------------
Column: auto_renewal_score
auto_renewal_score
8    49575
9    28750
Name: count, dtype: int64
Null values: 0
--------------------------------------------------
Column: status_scores
status_scores
9    46080
0    11791
8    11577
7     8877
Name: count, dtype: int64
Null values: 0
--------------------------------------------------
Column: anchoring_score
anchoring_score
9.5    28019
8.5    22037
7.5    15370
9.0    12898
8.0        1
Name: count, dtype: int64
Null values: 0
--------------------------------------------------
Column: tenure_scores
tenure_scores
9.5    43387
8.5    13612
9.0    10311
8.0   

Null values: 0
--------------------------------------------------
Column: call_number
call_number
2.0     16823
1.0     14241
3.0     13346
4.0      9584
5.0      6490
        ...  
58.0        1
42.0        1
51.0        1
49.0        1
54.0        1
Name: count, Length: 61, dtype: int64
Null values: 240
--------------------------------------------------
Column: crm_contractor_sentiment_score
crm_contractor_sentiment_score
 50.0     27170
-1.0      21710
 80.0      5039
 20.0      2235
 90.0      1439
 30.0       894
 40.0       742
 0.0        651
 100.0      507
 16.0        18
 98.0        12
 84.0         9
 48.0         5
 55.0         5
 60.0         3
 31.0         3
 25.5         2
 70.0         1
 91.0         1
Name: count, dtype: int64
Null values: 17879
--------------------------------------------------
Column: crm_agent_chase_count
crm_agent_chase_count
1.0     24479
0.0     16643
2.0     14782
3.0      3661
4.0       653
5.0       160
6.0        50
7.0        10
8.0     

In [19]:
def fill_numericals_refined(df, num_cols):
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            
            # 1. Logical Zero Fill: These columns likely mean "None" if NaN
            zero_fill = ['call_number', 'crm_agent_chase_count', 'crm_auto_renewal_status']
            if any(x in col for x in zero_fill):
                df[col] = df[col].fillna(0)
                continue

            # 2. Sentiment Score Logic: Neutral point is 50.0
            # If sentiment is missing, we shouldn't assume perfection (90) or failure (0).
            # Filling with the median is safest for these specific distributions.
            sentiment_cols = ['sentiment_score', 'sentiment_start', 'sentiment_end', 'sentiment_overall']
            if any(x in col for x in sentiment_cols):
                df[col] = df[col].fillna(df[col].median())
                continue

            # 3. General Distribution Logic for any other columns
            skewness = df[col].skew()
            if abs(skewness) > 1:
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(df[col].mean())

In [20]:
fill_numericals_refined(df_filtered, num_cols)

In [21]:
for col in df_filtered.columns:
    if df_filtered[col].dtype in ['int64', 'float64']:
        print(f"Column: {col}")
        print(df_filtered[col].value_counts())
        print(f"Null values: {df_filtered[col].isnull().sum()}")
        print("-" * 50)

Column: sustainability_score
sustainability_score
8.0    44439
9.5    24068
9.0     9818
Name: count, dtype: int64
Null values: 0
--------------------------------------------------
Column: total_renewal_score_new
total_renewal_score_new
44.0    9954
43.0    7409
45.5    7115
45.0    6731
44.5    6652
43.5    6314
42.0    5026
42.5    3345
41.0    3198
41.5    2544
46.5    2364
32.0    1972
33.0    1883
46.0    1783
34.0    1454
32.5    1439
40.5    1438
40.0    1397
35.0    1125
33.5    1007
34.5     937
31.0     937
39.5     754
31.5     704
39.0     315
38.5     166
36.0     133
35.5      77
30.5      59
36.5      55
38.0      29
37.5       8
37.0       1
Name: count, dtype: int64
Null values: 0
--------------------------------------------------
Column: last_years_price
last_years_price
599.00     3454
999.00     3427
799.00     2901
949.00     2341
579.00     2242
           ... 
702.00        1
813.60        1
833.40        1
1592.10       1
258.75        1
Name: count, Length: 141

In [22]:
df_filtered = df_filtered.fillna('Unknown')

In [23]:
df_filtered.shape

(78325, 110)

In [24]:
for col in df_filtered.columns:
    print(df_filtered[col].value_counts(dropna=False))
    print(df_filtered[col].isnull().sum())
    print("-" * 50)

sustainability_score
8.0    44439
9.5    24068
9.0     9818
Name: count, dtype: int64
0
--------------------------------------------------
total_renewal_score_new
44.0    9954
43.0    7409
45.5    7115
45.0    6731
44.5    6652
43.5    6314
42.0    5026
42.5    3345
41.0    3198
41.5    2544
46.5    2364
32.0    1972
33.0    1883
46.0    1783
34.0    1454
32.5    1439
40.5    1438
40.0    1397
35.0    1125
33.5    1007
34.5     937
31.0     937
39.5     754
31.5     704
39.0     315
38.5     166
36.0     133
35.5      77
30.5      59
36.5      55
38.0      29
37.5       8
37.0       1
Name: count, dtype: int64
0
--------------------------------------------------
last_years_price
599.00     3454
999.00     3427
799.00     2901
949.00     2341
579.00     2242
           ... 
702.00        1
813.60        1
833.40        1
1592.10       1
258.75        1
Name: count, Length: 1416, dtype: int64
0
--------------------------------------------------
auto_renewal_score
8    49575
9    28750
Na

0
--------------------------------------------------
call_number
2.0     16823
1.0     14241
3.0     13346
4.0      9584
5.0      6490
        ...  
58.0        1
42.0        1
51.0        1
49.0        1
54.0        1
Name: count, Length: 62, dtype: int64
0
--------------------------------------------------
time_to_renewal
pre_renewal    60446
Unknown        17879
Name: count, dtype: int64
0
--------------------------------------------------
crm_accreditation_completed
Not Discussed    31469
Unknown          23440
No               16443
Yes               6973
Name: count, dtype: int64
0
--------------------------------------------------
crm_timely_completion
Not Discussed    48628
Unknown          23440
No                4679
Yes               1578
Name: count, dtype: int64
0
--------------------------------------------------
crm_progress_towards_accreditation
Not Discussed    40655
Unknown          23440
Yes              12793
No                1437
Name: count, dtype: int64
0
------

In [25]:
for col in df_filtered.columns:
    print(col)

sustainability_score
total_renewal_score_new
last_years_price
auto_renewal_score
status_scores
anchoring_score
tenure_scores
proforma_auto_renewal
current_anchorings
payment_timeframe
current_auto_renewal_flag
renewal_score_at_release
proforma_membership_status
tenure_years
band
starting_net
starting_membership_net
starting_package_net
starting_pqq_net
membership_net
package_net
pqqnet
prospect_outcome
payment_method
total_amount
tenure_group
last_renewal
last_band
last_total_net_paid
last_connections
anchor_group
is_first_year
has_discount
discount_pct
call_direction
membership_renewal_decision
serious_complaint
other_complaint
discussion_on_price_increase
renewal_impact_due_to_price_increase
discount_or_waiver_requested
call_reschedule_request
agent_flagged_membership_status_alert
agent_renewal_initiation
explicit_competitor_mention
explicit_switching_intent
mentioned_competitors
price_switching_mentioned
competitor_value_comparison
competitor_benefits_mentioned
topic_introduced_by
p

In [26]:
df_filtered.info()

<class 'pandas.DataFrame'>
Index: 78325 entries, 6 to 665981
Columns: 110 entries, sustainability_score to cc_contractor_complained
dtypes: bool(1), float64(18), int64(13), str(78)
memory usage: 65.8 MB


In [27]:
df_filtered.head()

,sustainability_score,total_renewal_score_new,last_years_price,auto_renewal_score,status_scores,anchoring_score,tenure_scores,proforma_auto_renewal,current_anchorings,payment_timeframe,current_auto_renewal_flag,renewal_score_at_release,proforma_membership_status,tenure_years,band,starting_net,starting_membership_net,starting_package_net,starting_pqq_net,membership_net,package_net,pqqnet,prospect_outcome,payment_method,total_amount,tenure_group,last_renewal,last_band,last_total_net_paid,last_connections,anchor_group,is_first_year,has_discount,discount_pct,call_direction,membership_renewal_decision,serious_complaint,other_complaint,discussion_on_price_increase,renewal_impact_due_to_price_increase,discount_or_waiver_requested,call_reschedule_request,agent_flagged_membership_status_alert,agent_renewal_initiation,explicit_competitor_mention,explicit_switching_intent,mentioned_competitors,price_switching_mentioned,competitor_value_comparison,competitor_benefits_mentioned,topic_introduced_by,percentage_price_increase_mentioned,monetary_price_increase_mentioned,price_range_mentioned,customer_asked_for_justification,customer_response,desire_to_cancel,discount_offered,call_number,time_to_renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,crm_dts_or_ssip_mentioned,crm_customer_payment_intention,crm_competitors_mentioned,crm_membership_level,crm_platform_issues_raised,crm_agent_chased_contractor,crm_agent_chase_count,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_call_initiated_by,cc_questionnaire_completion,cc_chasing_response,cc_issues_within_questionnaire,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_contractor_sentiment,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained
6,8.0,41.5,799.0,9,9,7.5,8.0,True,0,0,y,24.5,Accredited,1,Band C1,919,664,120,135,664.0,0.0,135,Won,Card,799.0,1,2025-08-09,Band C1,919.0,0,1,1,0,0.0,OUT_BOUND,No,No,No,No,No,No,No,No,Yes,No,No,No,No,Not Applicable,Not Applicable,Not Discussed,No,No,Not Discussed,No,Neutral,Renewed,No,1.0,pre_renewal,No,Not Discussed,Yes,No,No,Yes,neutral,50.0,No,Not Discussed,No,Accredited,No,Yes,1.0,No,Yes,0.0,Yes,No,No,No,No,Not Discussed,Express,Yes,No,No,No,No,No,Agent,No,No,No,No,No,No,No,No,No,Satisfied,50.0,90.0,85.0,Yes,Yes,No,No,No
8,8.0,41.5,799.0,9,9,7.5,8.0,True,0,0,y,24.5,Accredited,1,Band C1,919,664,120,135,664.0,0.0,135,Won,Card,799.0,1,2025-08-09,Band C1,919.0,0,1,1,0,0.0,OUT_BOUND,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,2.0,pre_renewal,No,Not Discussed,Yes,No,No,Yes,neutral,50.0,No,Not Discussed,No,Accredited,No,Yes,1.0,No,Yes,0.0,Yes,No,No,No,No,Not Discussed,Express,Yes,No,No,No,No,No,Agent,No,No,Unknown,No,Yes,No,No,No,No,Satisfied,80.0,90.0,80.0,No,No,No,No,No
9,8.0,41.5,799.0,9,9,7.5,8.0,True,0,0,y,24.5,Accredited,1,Band C1,919,664,120,135,664.0,0.0,135,Won,Card,799.0,1,2025-08-09,Band C1,919.0,0,1,1,0,0.0,OUT_BOUND,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,U

In [28]:
categorical_cols = [col for col in df_filtered.columns if col not in num_cols]

In [29]:
print(len(categorical_cols))

79


In [30]:
for col in categorical_cols:
    print(f"Column: {col}")
    print(df_filtered[col].nunique())
    print(df_filtered[col].isnull().sum())
    print("-" * 50)

Column: proforma_auto_renewal
2
0
--------------------------------------------------
Column: current_auto_renewal_flag
2
0
--------------------------------------------------
Column: proforma_membership_status
3
0
--------------------------------------------------
Column: band
13
0
--------------------------------------------------
Column: prospect_outcome
2
0
--------------------------------------------------
Column: payment_method
5
0
--------------------------------------------------
Column: tenure_group
4
0
--------------------------------------------------
Column: last_renewal
558
0
--------------------------------------------------
Column: last_band
14
0
--------------------------------------------------
Column: anchor_group
6
0
--------------------------------------------------
Column: call_direction
3
0
--------------------------------------------------
Column: membership_renewal_decision
3
0
--------------------------------------------------
Column: serious_complaint
3
0
------

0
--------------------------------------------------
Column: topic_introduced_by


4
0
--------------------------------------------------
Column: percentage_price_increase_mentioned
4
0
--------------------------------------------------
Column: monetary_price_increase_mentioned
4
0
--------------------------------------------------
Column: price_range_mentioned
765
0
--------------------------------------------------
Column: customer_asked_for_justification
4
0
--------------------------------------------------
Column: customer_response
5
0
--------------------------------------------------
Column: desire_to_cancel
4
0
--------------------------------------------------
Column: discount_offered
4
0
--------------------------------------------------
Column: time_to_renewal
2
0
--------------------------------------------------
Column: crm_accreditation_completed
4
0
--------------------------------------------------
Column: crm_timely_completion
4
0
--------------------------------------------------
Column: crm_progress_towards_accreditation
4


0
--------------------------------------------------
Column: crm_delays_in_accreditation
4
0
--------------------------------------------------
Column: crm_contractor_suggested_leave


4
0
--------------------------------------------------
Column: crm_contractor_engagement
3
0
--------------------------------------------------
Column: crm_contractor_sentiment
4
0
--------------------------------------------------
Column: crm_dts_or_ssip_mentioned
4
0
--------------------------------------------------
Column: crm_customer_payment_intention
4
0
--------------------------------------------------
Column: crm_competitors_mentioned
4
0
--------------------------------------------------
Column: crm_membership_level
8
0
--------------------------------------------------
Column: crm_platform_issues_raised
4
0
--------------------------------------------------
Column: crm_agent_chased_contractor
4
0
--------------------------------------------------
Column: crm_accreditation_issues
4
0
--------------------------------------------------
Column: crm_membership_overdue
4
0
--------------------------------------------------
Column: crm_dissatisified_with_renewal_price
4
0
--------

3
0
--------------------------------------------------
Column: cc_call_initiated_by
4
0
--------------------------------------------------
Column: cc_questionnaire_completion
3
0
--------------------------------------------------
Column: cc_chasing_response
3
0
--------------------------------------------------
Column: cc_issues_within_questionnaire
3
0
--------------------------------------------------
Column: cc_login_issues
3
0
--------------------------------------------------
Column: cc_platform_issues
3
0
--------------------------------------------------
Column: cc_dissatisfaction_time_to_complete
3
0
--------------------------------------------------
Column: cc_process_complexity_concerns
3
0
--------------------------------------------------
Column: cc_questions_harder_than_expected
3
0
--------------------------------------------------
Column: cc_dissatisfaction_support


3
0
--------------------------------------------------
Column: cc_contractor_sentiment
5


0
--------------------------------------------------
Column: cc_pricing_mentioned
3
0
--------------------------------------------------
Column: cc_pricing_sentiment_impact
3
0
--------------------------------------------------
Column: cc_refund_discussed
3
0
--------------------------------------------------
Column: cc_contractor_suggest_leave
3
0
--------------------------------------------------
Column: cc_contractor_complained
3
0
--------------------------------------------------


In [31]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

def preprocess_and_encode(df):
    """
    Preprocesses the dataframe by:
    1. Removing date columns (those with 'date' in name or datetime dtype).
    2. Encoding categorical columns based on unique value count:
       - 2 uniques: Label encoding
       - 3-10 uniques: One-hot encoding (drop_first=True)
       - >10 uniques: Frequency encoding
    
    Args:
        df (pd.DataFrame): Input dataframe
    
    Returns:
        pd.DataFrame: Processed dataframe with dates removed and categoricals encoded
    """
    # Step 1: Identify and remove date columns
    date_cols = [
        col for col in df.columns 
        if 'date' in col.lower() or df[col].dtype == 'datetime64[ns]'
    ]
    df = df.drop(columns=date_cols, errors='ignore')
    print(f"Removed date columns: {date_cols}")
    
    # Step 2: Identify categorical columns
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    
    # Step 3: Encode categoricals
    for col in cat_cols:
        n_unique = df[col].nunique()
        
        if n_unique <= 3:
            # Label encoding for binary
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            print(f"Label encoded: {col} ({n_unique} uniques)")
            
        elif 4 <= n_unique <= 10:
            # One-hot encoding
            df = pd.get_dummies(df, columns=[col], drop_first=True)
            print(f"One-hot encoded: {col} ({n_unique} uniques)")
            
        else:
            # Frequency encoding for high cardinality
            freq_map = df[col].value_counts()
            df[col] = df[col].map(freq_map)
            print(f"Frequency encoded: {col} ({n_unique} uniques)")
    
    print(f"Final shape: {df.shape}")
    return df

In [32]:
df_encoded = preprocess_and_encode(df_filtered.copy())

Removed date columns: []
Label encoded: current_auto_renewal_flag (2 uniques)
Label encoded: proforma_membership_status (3 uniques)
Frequency encoded: band (13 uniques)
Label encoded: prospect_outcome (2 uniques)
One-hot encoded: payment_method (5 uniques)
One-hot encoded: tenure_group (4 uniques)
Frequency encoded: last_renewal (558 uniques)
Frequency encoded: last_band (14 uniques)
One-hot encoded: anchor_group (6 uniques)
Label encoded: call_direction (3 uniques)
Label encoded: membership_renewal_decision (3 uniques)
Label encoded: serious_complaint (3 uniques)
Label encoded: other_complaint (3 uniques)
Label encoded: discussion_on_price_increase (3 uniques)
Label encoded: renewal_impact_due_to_price_increase (3 uniques)
Label encoded: discount_or_waiver_requested (3 uniques)
Label encoded: call_reschedule_request (3 uniques)
Label encoded: agent_flagged_membership_status_alert (3 uniques)
Label encoded: agent_renewal_initiation (3 uniques)
Label encoded: explicit_competitor_mention

C:\Users\Gaurav Garg\AppData\Local\Temp\ipykernel_14808\3224478787.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns


Label encoded: mentioned_competitors (3 uniques)
Label encoded: price_switching_mentioned (3 uniques)
One-hot encoded: competitor_value_comparison (5 uniques)
One-hot encoded: competitor_benefits_mentioned (5 uniques)
One-hot encoded: topic_introduced_by (4 uniques)
One-hot encoded: percentage_price_increase_mentioned (4 uniques)
One-hot encoded: monetary_price_increase_mentioned (4 uniques)
Frequency encoded: price_range_mentioned (765 uniques)
One-hot encoded: customer_asked_for_justification (4 uniques)
One-hot encoded: customer_response (5 uniques)
One-hot encoded: desire_to_cancel (4 uniques)
One-hot encoded: discount_offered (4 uniques)


Label encoded: time_to_renewal (2 uniques)
One-hot encoded: crm_accreditation_completed (4 uniques)
One-hot encoded: crm_timely_completion (4 uniques)
One-hot encoded: crm_progress_towards_accreditation (4 uniques)
One-hot encoded: crm_delays_in_accreditation (4 uniques)
One-hot encoded: crm_contractor_suggested_leave (4 uniques)
Label encoded: crm_contractor_engagement (3 uniques)
One-hot encoded: crm_contractor_sentiment (4 uniques)
One-hot encoded: crm_dts_or_ssip_mentioned (4 uniques)
One-hot encoded: crm_customer_payment_intention (4 uniques)
One-hot encoded: crm_competitors_mentioned (4 uniques)


One-hot encoded: crm_membership_level (8 uniques)
One-hot encoded: crm_platform_issues_raised (4 uniques)
One-hot encoded: crm_agent_chased_contractor (4 uniques)
One-hot encoded: crm_accreditation_issues (4 uniques)
One-hot encoded: crm_membership_overdue (4 uniques)
One-hot encoded: crm_dissatisified_with_renewal_price (4 uniques)
Label encoded: crm_customer_complained (3 uniques)
Label encoded: crm_refund_mentioned (3 uniques)
One-hot encoded: crm_negative_customer_experience (4 uniques)
One-hot encoded: crm_dissatisfaction_with_support (4 uniques)
One-hot encoded: crm_financial_hardship_mentioned (4 uniques)


One-hot encoded: cc_care_package (6 uniques)
Label encoded: cc_care_package_discussed (3 uniques)
Label encoded: cc_urgency_getting_on_site (3 uniques)
Label encoded: cc_external_consultant (3 uniques)
Label encoded: cc_agent_cross_sell_attempt (3 uniques)
Label encoded: cc_customer_issues_concerns (3 uniques)
Label encoded: cc_business_struggles_financial_hardship (3 uniques)
One-hot encoded: cc_call_initiated_by (4 uniques)
Label encoded: cc_questionnaire_completion (3 uniques)
Label encoded: cc_chasing_response (3 uniques)


Label encoded: cc_issues_within_questionnaire (3 uniques)
Label encoded: cc_login_issues (3 uniques)
Label encoded: cc_platform_issues (3 uniques)
Label encoded: cc_dissatisfaction_time_to_complete (3 uniques)
Label encoded: cc_process_complexity_concerns (3 uniques)
Label encoded: cc_questions_harder_than_expected (3 uniques)
Label encoded: cc_dissatisfaction_support (3 uniques)
One-hot encoded: cc_contractor_sentiment (5 uniques)
Label encoded: cc_pricing_mentioned (3 uniques)
Label encoded: cc_pricing_sentiment_impact (3 uniques)
Label encoded: cc_refund_discussed (3 uniques)
Label encoded: cc_contractor_suggest_leave (3 uniques)


Label encoded: cc_contractor_complained (3 uniques)
Final shape: (78325, 189)


In [33]:
# Check the result
print(df_encoded.head())
print(df_encoded.info())

    sustainability_score  total_renewal_score_new  last_years_price  \
6                    8.0                     41.5             799.0   
8                    8.0                     41.5             799.0   
9                    8.0                     41.5             799.0   
10                   8.0                     41.5             799.0   
11                   8.0                     33.0             799.0   

    auto_renewal_score  status_scores  anchoring_score  tenure_scores  \
6                    9              9              7.5            8.0   
8                    9              9              7.5            8.0   
9                    9              9              7.5            8.0   
10                   9              9              7.5            8.0   
11                   8              0              7.5            9.5   

    proforma_auto_renewal  current_anchorings  payment_timeframe  \
6                    True                   0                  0  

In [34]:
df_encoded

,sustainability_score,total_renewal_score_new,last_years_price,auto_renewal_score,status_scores,anchoring_score,tenure_scores,proforma_auto_renewal,current_anchorings,payment_timeframe,current_auto_renewal_flag,renewal_score_at_release,proforma_membership_status,tenure_years,band,starting_net,starting_membership_net,starting_package_net,starting_pqq_net,membership_net,package_net,pqqnet,prospect_outcome,total_amount,last_renewal,last_band,last_total_net_paid,last_connections,is_first_year,has_discount,discount_pct,call_direction,membership_renewal_decision,serious_complaint,other_complaint,discussion_on_price_increase,renewal_impact_due_to_price_increase,discount_or_waiver_requested,call_reschedule_request,agent_flagged_membership_status_alert,agent_renewal_initiation,explicit_competitor_mention,explicit_switching_intent,mentioned_competitors,price_switching_mentioned,price_range_mentioned,call_number,time_to_renewal,crm_contractor_engagement,crm_contractor_sentiment_score,crm_agent_chase_count,crm_auto_renewal_status,crm_customer_complained,crm_refund_mentioned,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_questionnaire_completion,cc_chasing_response,cc_issues_within_questionnaire,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,payment_method_Card,payment_method_Cheque,payment_method_Unknown,payment_method_World Pay,tenure_group_2,tenure_group_3,tenure_group_4+,anchor_group_10+,anchor_group_2,anchor_group_3,anchor_group_4 to 9,anchor_group_Independent,competitor_value_comparison_Not Applicable,competitor_value_comparison_Not Discussed,competitor_value_comparison_Similar Value,competitor_value_comparison_Unknown,competitor_benefits_mentioned_Discounts,competitor_benefits_mentioned_Not Applicable,competitor_benefits_mentioned_Not Discussed,competitor_benefits_mentioned_Unknown,topic_introduced_by_Customer,topic_introduced_by_Not Discussed,topic_introduced_by_Unknown,percentage_price_increase_mentioned_Not Discussed,percentage_price_increase_mentioned_Unknown,percentage_price_increase_mentioned_Yes,monetary_price_increase_mentioned_Not Discussed,monetary_price_increase_mentioned_Unknown,monetary_price_increase_mentioned_Yes,customer_asked_for_justification_Not Discussed,customer_asked_for_justification_Unknown,customer_asked_for_justification_Yes,customer_response_Neutral,customer_response_Not Discussed,customer_response_Positive,customer_response_Unknown,desire_to_cancel_Not Discussed,desire_to_cancel_Renewed,desire_to_cancel_Unknown,discount_offered_Not Discussed,discount_offered_Unknown,discount_offered_Yes,crm_accreditation_completed_Not Discussed,crm_accreditation_completed_Unknown,crm_accreditation_completed_Yes,crm_timely_completion_Not Discussed,crm_timely_completion_Unknown,crm_timely_completion_Yes,crm_progress_towards_accreditation_Not Discussed,crm_progress_towards_accreditation_Unknown,crm_progress_towards_accreditation_Yes,crm_delays_in_accreditation_Not Discussed,crm_delays_in_accreditation_Unknown,crm_delays_in_accreditation_Yes,crm_contractor_suggested_leave_Not Discussed,crm_contractor_suggested_leave_Unknown,crm_contractor_suggested_leave_Yes,crm_contractor_sentiment_dissatisfied,crm_contractor_sentiment_neutral,crm_contractor_sentiment_satisfied,crm_dts_or_ssip_mentioned_Not Discussed,crm_dts_or_ssip_mentioned_Unknown,crm_dts_or_ssip_mentioned_Yes,crm_customer_payment_intention_Not Discussed,crm_customer_payment_intention_Unknown,crm_customer_payment_intention_Yes,crm_competitors_mentioned_Not Discussed,crm_competitors_mentioned_Unknown,crm_co

In [35]:
df_encoded.to_csv('../../../data/processed/churn_prediction_dataset.csv', index=False)

# Finalizing Datasets for Next Steps


In [36]:
# Using open_prospects created earlier
df_open_encoded = None
open_refs = open_prospects['co_ref'].copy()
df_open = open_prospects.drop(columns=[c for c in columns_to_drop if c in open_prospects.columns])
num_cols_open = [c for c in df_open.columns if df_open[c].dtype in ['int64', 'float64']]
fill_numericals_refined(df_open, num_cols_open)
df_open = df_open.fillna('Unknown')
    
import sys, os
old_stdout = sys.stdout
sys.stdout = open(os.devnull, 'w')
df_open_encoded = preprocess_and_encode(df_open.copy())
sys.stdout = old_stdout
df_open_encoded['co_ref'] = open_refs.values
    
# Recreate the engineered features on open_prospects to perfectly match df_encoded
df_open_encoded['price_change'] = df_open_encoded['total_amount'] - df_open_encoded['last_years_price']
df_open_encoded['price_change_pct'] = np.where(df_open_encoded['last_years_price'] > 0, (df_open_encoded['total_amount'] - df_open_encoded['last_years_price']) / df_open_encoded['last_years_price'] * 100, 0)
df_open_encoded['price_increase_flag'] = (df_open_encoded['price_change'] > 0).astype(int)
if 'cc_contractor_sentiment_end_score' in df_open_encoded.columns and 'cc_contractor_sentiment_start_score' in df_open_encoded.columns:
    df_open_encoded['cc_sentiment_delta'] = df_open_encoded['cc_contractor_sentiment_end_score'] - df_open_encoded['cc_contractor_sentiment_start_score']
    df_open_encoded['cc_sentiment_improved'] = (df_open_encoded['cc_sentiment_delta'] > 0).astype(int)
    df_open_encoded['cc_sentiment_declined'] = (df_open_encoded['cc_sentiment_delta'] < 0).astype(int)
df_open_encoded['complaint_intensity'] = df_open_encoded.get('serious_complaint', 0) + df_open_encoded.get('other_complaint', 0) + df_open_encoded.get('cc_contractor_complained', 0) + df_open_encoded.get('crm_customer_complained', 0)
df_open_encoded['dissatisfaction_composite'] = df_open_encoded.get('cc_dissatisfaction_time_to_complete', 0) + df_open_encoded.get('cc_dissatisfaction_support', 0) + df_open_encoded.get('cc_process_complexity_concerns', 0) + df_open_encoded.get('cc_platform_issues', 0)
df_open_encoded['competitive_pressure'] = df_open_encoded.get('explicit_competitor_mention', 0) + df_open_encoded.get('mentioned_competitors', 0) + df_open_encoded.get('price_switching_mentioned', 0)
df_open_encoded['payment_delay_flag'] = (df_open_encoded.get('payment_timeframe', 0) < 0).astype(int)
df_open_encoded['abs_payment_timeframe'] = df_open_encoded.get('payment_timeframe', 0).abs()
df_open_encoded['tenure_price_ratio'] = df_open_encoded['total_amount'] / (df_open_encoded.get('tenure_years', 0) + 1)
df_open_encoded['pqq_share'] = np.where(df_open_encoded['total_amount'] > 0, df_open_encoded.get('pqqnet', 0) / df_open_encoded['total_amount'], 0)
df_open_encoded['package_share'] = np.where(df_open_encoded['total_amount'] > 0, df_open_encoded.get('package_net', 0) / df_open_encoded['total_amount'], 0)

df_open_encoded.to_csv('../../../data/processed/open_prospects_dataset.csv', index=False)
print('Saved open prospects dataset to: data/processed/open_prospects_dataset.csv')


C:\Users\Gaurav Garg\AppData\Local\Temp\ipykernel_14808\3224478787.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns


Saved open prospects dataset to: data/processed/open_prospects_dataset.csv
